# ESCREVER INTRO AQUI

In [ ]:
from types import SimpleNamespace
from cipher_base import BLOCK_SIZE
from ctr import random_nonce, apply_ctr
from feistel import Feistel80
from spn import SPN80

BLOCK_BYTES = BLOCK_SIZE // 8

In [ ]:
MASTER_KEY = 0x4a8f3c11e792d504b86a
CIPHERS = {
	"S07": {
		"SBOX": [15 - x for x in range(16)],
		"SPN_PBOX": [(29 * i + 7) % 80 for i in range(80)],
		"FEISTEL_PBOX": [(17 * i + 9) % 40 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	},
	"F01": {
		"SBOX": [(x*x + 3) % 16 for x in range(16)],
		"SPN_PBOX": [i ^ 1 for i in range(80)],
		"FEISTEL_PBOX": [i ^ 1 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	},
	"F07": {
		"SBOX": [15 - x for x in range(16)],
		"SPN_PBOX": [(29 * i + 7) % 80 for i in range(80)],
		"FEISTEL_PBOX": [(17 * i + 9) % 40 for i in range(40)],
		"MASTER_KEY": MASTER_KEY
	}
}

In [ ]:
def avalanche_effect(cipher: SPN80 | Feistel80, verbose = False) -> float:
	messages = [(1 << i) - 1 for i in range(1, 81)]

	prev_enc = cipher.encrypt_block(0)
	total_bits_diff = 0

	for index, data in enumerate(messages, start=1):
		ciphertext = cipher.encrypt_block(data)
		diff = ciphertext ^ prev_enc
		total_bits_diff += diff.bit_count()
		prev_enc = ciphertext

		if verbose:
			print()
			print(f"message {index}")
			print(f"  {'hex':11s}  0x{data:0{BLOCK_BYTES * 2}x}")
			print(f"  {'enc':11s}  0x{ciphertext:0{BLOCK_BYTES * 2}x}")
			print(f"  {'dif':11s}  0x{diff:0{BLOCK_BYTES * 2}x}")

	return total_bits_diff/len(messages)

In [ ]:
print("Avalanche Effect Analysis")
for key, design in CIPHERS.items():
	print(key)

	mode = ('SPN' if key[0] == 'S' else 'Feistel')
	cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

	print(f"{'cipher:':8s}{mode}, {cipher.rounds} rounds")

	avg_bits_flipped = avalanche_effect(cipher)

	print(f"avg diff is {avg_bits_flipped} bit(s) per plaintext bit flipped")
	print(f"avg expected diff is {BLOCK_SIZE // 2} bit(s) per plaintext bit flipped")
	print()


In [ ]:
long_message = "Tudo o que eu percebo no mundo, percebo em mim. Isto nada mais é do que o meu corpo afetado pelo mundo. Se você vai ter que conviver com você mesmo até o fim, se você vai ter que se aguentar até o fim, se você vai ser espectador de você mesmo até o fim, é melhor que se encante com o que faz!"

print(f"0x{long_message.encode().hex()}")

In [ ]:
def nistRunsTest(bits_string: str) -> bool:
	from math import sqrt, erfc

	def abs(x: float) -> float:
		return x if x >= 0 else -x

	bits = bits_string[2:]
	n = len(bits)

	pi = 0
	for bit in bits:
		pi += int(bit)
	pi /= n

	if abs(pi - 0.5) >= (2/sqrt(n)):
		return False

	v_obs = 1
	for i in range(len(bits)-1):
		if bits[i] == bits[i+1]:
			v_obs += 1

	p_value = erfc((abs(v_obs - (2*n*pi*(1-pi))))/((2*sqrt(2*n)*pi*(1-pi))))
	
	return p_value >= 0.01

In [ ]:
print("NIST - 2.3 Runs Test\n")
for key, design in CIPHERS.items():
	print(key)

	mode = ('SPN' if key[0] == 'S' else 'Feistel')
	cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

	print(f"{'cipher:':8s}{mode}, {cipher.rounds} rounds")

	message = long_message.encode()
	message_hex = f"0x{message.hex()}"
	nonce = random_nonce()

	ciphertext = apply_ctr(cipher, nonce, message)
	ciphertext_bin = f"0b{(bin(int.from_bytes(ciphertext))[2:]).zfill(2416)}"
	ciphertext_hex = f"0x{ciphertext.hex()}"


	print(f"message '{long_message}'")
	print(f"  {'hex':11s}  {message_hex}")
	print(f"  {'enc':11s}  {ciphertext_hex}")

	is_random = nistRunsTest(ciphertext_bin)

	print(f"The sequence{'' if is_random else ' not'} is random\n")

In [ ]:
def nistCusumTest(bits_string: str, mode: 0 | 1 = 0) -> bool:
	from math import sqrt, erfc

	def phi(x: float) -> float:
		return 0.5 * erfc(-x / sqrt(2))

	bits = bits_string[2:]
	n = len(bits)

	X = []
	for bit in bits:
		if bit == '0':
			X.append(-1)
		else:
			X.append(1)

	cusum = X.copy()

	if mode == 1:
		cusum = cusum[::-1]

	for i in range(1, len(X)):
		cusum[i] += cusum[i-1]

	z = max(cusum)

	z_sqrtN = z/sqrt(n)

	p_value = 1
	lw = ((-n/z)-3)/4
	md = ((-n/z)+1)/4
	up = ((n/z)-1)/4

	for k in range(md, up+1):
		p_value -= (phi(((4*k)+1)*z_sqrtN) - phi(((4*k)-1)*z_sqrtN))

	for k in range(lw, up+1):
		p_value += (phi(((4*k)+3)*z_sqrtN) - phi(((4*k)+1)*z_sqrtN))
	
	return p_value >= 0.01

In [ ]:
print("NIST - 2.13 Cumulative Sums (Cusum) Test\n")
for key, design in CIPHERS.items():
	print(key)

	mode = ('SPN' if key[0] == 'S' else 'Feistel')
	cipher = (SPN80 if mode == 'SPN' else Feistel80).from_design(SimpleNamespace(**design))

	print(f"{'cipher:':8s}{mode}, {cipher.rounds} rounds")

	message = long_message.encode()
	message_hex = f"0x{message.hex()}"
	nonce = random_nonce()

	ciphertext = apply_ctr(cipher, nonce, message)
	ciphertext_bin = f"0b{(bin(int.from_bytes(ciphertext))[2:]).zfill(2416)}"
	ciphertext_hex = f"0x{ciphertext.hex()}"


	print(f"message '{long_message}'")
	print(f"  {'hex':11s}  {message_hex}")
	print(f"  {'enc':11s}  {ciphertext_hex}")

	is_random = nistCusumTest(ciphertext_bin)

	print(f"The sequence{'' if is_random else ' not'} is random\n")